# Week5_ex3 - Linear Motor Pulsating Current (Time Sweep)

Ansys Electronics Desktop Student doesn't support the Transient solver at all, and this exercise was originally built on a Maxwell 2D Transient model with a translational motion band. The mover never actually moves here (velocity is 0), and only the A-phase current changes over time, so I dropped the motion/band setup, switched to Magnetostatic, and swept a `t_sample` design variable through one electrical cycle with a parametric sweep instead of a real time axis. The force and current end up as discrete samples across the cycle rather than a continuous waveform, but the pulsating-current behavior this exercise is about still comes through.

In [ ]:
import pandas as pd
import numpy as np
import os
import ansys.aedt.core
import math
import shutil
import time
import matplotlib.pyplot as plt

In [ ]:
# 1. Launch AEDT
DT = ansys.aedt.core.Desktop(version="2025.2", non_graphical=False, student_version=True)

# 2. Turn off autosave
DT.disable_autosave()

# 3. Create project and set solution type
sol_type = "Magnetostatic"
M2D = ansys.aedt.core.maxwell.Maxwell2d(solution_type=sol_type)

# 4. odesign object for script-recording style calls
oDesign = M2D.odesign

In [ ]:
## Set up output directory for results ##

proj_name = "Week5_ex3"

# If ANSYS_PROJECT_DIR is set on this machine, save there.
# Otherwise, fall back to the notebook's own working directory,
# so this stays portable for anyone else running the notebook as-is.
base_dir = os.environ.get("ANSYS_PROJECT_DIR", os.getcwd())
dir = os.path.join(base_dir, proj_name)
print(dir)
os.makedirs(dir, exist_ok=True)

desi_name = "Week5_ex3"

In [ ]:
## Save project and apply design name ##

proj = M2D.oproject
proj.SaveAs(f"{dir}\\{proj_name}.aedt", True)
M2D.rename_design(desi_name, save=False)

# Save again so the design-name change is committed to disk
M2D.save_project()

In [ ]:
depth = 40     # motor stack depth
M2D["depth"] = f"{depth}mm"

num_pole = 10    # number of magnet poles
M2D["num_pole"] = f"{num_pole}"

m_w = 20   # magnet width
M2D["m_w"] = f"{m_w}mm"

m_h = 5   # magnet height
M2D["m_h"] = f"{m_h}mm"

gap_w = m_w*0.3   # gap between adjacent magnets
M2D["gap_w"] = f"{gap_w}mm"

my_th = 15   # magnet yoke plate thickness
M2D["my_th"] = f"{my_th}mm"

num_slot = 3   # number of slots
M2D["num_slot"] = f"{num_slot}"

airgap = 1   # air gap length
M2D["airgap"] = f"{airgap}mm"

t_w = 6   # tooth width
M2D["t_w"] = f"{t_w}mm"

t_g = (52/3-6)/2   # tooth spacing
M2D["t_g"] = f"{t_g}mm"

J_rated = 3   # current density in the coil region
M2D["J_rated"] = f"{J_rated}"

cy_th = 15   # yoke thickness
M2D["cy_th"] = f"{cy_th}mm"

c_w = 5   # coil width
M2D["c_w"] = f"{c_w}mm"

c_h = 15   # coil height
M2D["c_h"] = f"{c_h}mm"

t_h = c_h   # tooth height
M2D["t_h"] = f"{t_h}mm"

my_w = (m_w * num_pole + gap_w * (num_pole - 1)) / 3 * 4   # magnet yoke plate width
M2D["my_w"] = f"{my_w}mm"


# current calculation

turn_c = 100    # number of coil turns
M2D["turn_c"] = f"{turn_c}"

area = c_w*c_h      # terminal cross-section area

current_rated = J_rated*area/turn_c    # rated phase current
M2D["current_rated"] = f"{current_rated}"

gamma = math.pi*0   # current angle offset (0 = no mismatch)
# gamma = math.pi/12   # 15 deg current angle mismatch
# gamma = math.pi/6   # 30 deg current angle mismatch
# gamma = math.pi/4   # 45 deg current angle mismatch
# gamma = math.pi/3   # 60 deg current angle mismatch
M2D["gamma"] = f"{gamma}"

theta_elec_offset = math.pi/2+gamma   # id=0 control angle, plus any offset above
M2D["theta_elec_offset"] = f"{theta_elec_offset}"

freq_elec = 10      # electrical frequency
M2D["freq_elec"] = f"{freq_elec}"


# The original notebook drove current_a with AEDT's built-in "Time" variable, which
# only exists under the Transient solver. Since Student doesn't support Transient at
# all, t_sample is a plain design variable instead -- swept below with a parametric
# sweep so each point gets solved as its own magnetostatic snapshot.
t_sample = 0
M2D["t_sample"] = f"{t_sample}s"

current_a = f"{current_rated}*cos(2*pi*{freq_elec}*t_sample + {theta_elec_offset})"  # pulsating current, swept over t_sample
# current_a = f"{current_rated}*cos({theta_elec_offset})"  # static current (no time dependence)
# current_a = "0"  # zero current

# current_b = f"{current_rated}*cos(2*pi*{freq_elec}*t_sample + pi*2/3 + {theta_elec_offset})"  # pulsating current
# current_b = f"{current_rated}*cos(pi*2/3 + {theta_elec_offset})"  # static current
current_b = "0"  # zero current

# current_c = f"{current_rated}*cos(2*pi*{freq_elec}*t_sample + pi*2/3*2 + {theta_elec_offset})"  # pulsating current
# current_c = f"{current_rated}*cos(pi*2/3*2 + {theta_elec_offset})"  # static current
current_c = "0"  # zero current

In [ ]:
steel = M2D.materials.duplicate_material("steel_1008", name="steel")  # iron core material


magnet_n = M2D.materials.duplicate_material("NdFe35", name="magnet_n")  # magnet with N pole facing +y
magnet_n.get_magnetic_coercivity()  # check current magnetization
magnet_n.set_magnetic_coercivity(value=-890000, x=0, y=1, z=0)  # magnetized along +y


magnet_s = M2D.materials.duplicate_material("NdFe35", name="magnet_s")  # magnet with S pole facing +y (magnetized -y)
magnet_s.get_magnetic_coercivity()  # check current magnetization
magnet_s.set_magnetic_coercivity(value=-890000, x=0, y=-1, z=0)  # magnetized along -y


copper_c = M2D.materials.duplicate_material("copper", name="copper_c")  # copper for the coils

In [ ]:
# 2D model depth
M2D.model_depth = "depth"

# Magnetostatic setup, nonlinear BH curve mode on
my_setup = M2D.create_setup(name="Setup1")
my_setup.props["MaximumPasses"] = 20
my_setup.props["MinimumPasses"] = 2
my_setup.props["MinimumConvergedPasses"] = 1
my_setup.props["PercentRefinement"] = 15
my_setup.props["PercentError"] = 1
my_setup.props["MuOption"] = {"MuNonLinearBH": True}
my_setup.update()

In [ ]:
# 1. Magnet yoke plate (substrate)
origin = ["-my_w/2", "-my_th/2"]
sizes = ["my_w", "my_th"]
substrate = M2D.modeler.create_rectangle(origin=origin, sizes=sizes, is_covered=True, name='substrate', material=steel, non_model=False)


# 2. Magnets
origin = [0, "-m_h/2"]
sizes = ["m_w", "m_h"]
magnet_1 = M2D.modeler.create_rectangle(origin=origin, sizes=sizes, is_covered=True, name="magnet_1", material=magnet_s, non_model=False)

vector = ["2*(m_w+gap_w)", 0]
M2D.modeler.duplicate_along_line(assignment=magnet_1, vector=vector, clones=num_pole/2, attach=True, is_3d_comp=False, duplicate_assignment=True)   # duplicate along the array

magnet_1.color = (0, 0, 255)


origin = ["gap_w+m_w", "-m_h/2"]
sizes = ["m_w", "m_h"]
magnet_2 = M2D.modeler.create_rectangle(origin=origin, sizes=sizes, is_covered=True, name="magnet_2", material=magnet_n, non_model=False)

vector = ["2*(m_w+gap_w)", 0]
M2D.modeler.duplicate_along_line(assignment=magnet_2, vector=vector, clones=num_pole/2, attach=True, is_3d_comp=False, duplicate_assignment=True)   # duplicate along the array

magnet_2.color = (255, 0, 0)


vector = ["-( m_w*num_pole+gap_w*(num_pole-1) )/2", "my_th/2+m_h/2"]
M2D.modeler.move(assignment=[magnet_1, magnet_2], vector=vector)    # center the array and lift it onto the yoke


M2D.modeler.separate_bodies(assignment=magnet_1, create_group=False)
magnet_1_1 = M2D.modeler.object_list[-4]
magnet_1_2 = M2D.modeler.object_list[-3]
magnet_1_3 = M2D.modeler.object_list[-2]
magnet_1_4 = M2D.modeler.object_list[-1]
M2D.modeler.separate_bodies(assignment=magnet_2, create_group=False)
magnet_2_1 = M2D.modeler.object_list[-4]
magnet_2_2 = M2D.modeler.object_list[-3]
magnet_2_3 = M2D.modeler.object_list[-2]
magnet_2_4 = M2D.modeler.object_list[-1]


# 3. Slots
origin = [0, "-t_h/2"]
sizes = ["t_w", "t_h"]
slot_a = M2D.modeler.create_rectangle(origin=origin, sizes=sizes, is_covered=True, name="slot_a", material="air", non_model=False)

vector=["t_w+2*t_g", 0]
M2D.modeler.duplicate_along_line(assignment=slot_a, vector=vector, clones=2, attach=False, is_3d_comp=False, duplicate_assignment=True)
slot_b = M2D.modeler.object_list[-1]
slot_b.name = "slot_b"

M2D.modeler.duplicate_along_line(assignment=slot_b, vector=vector, clones=2, attach=False, is_3d_comp=False, duplicate_assignment=True)
slot_c = M2D.modeler.object_list[-1]
slot_c.name = "slot_c"


# 4. Coils
origin = ["-c_w", "-t_h/2"]
sizes = ["t_w+2*c_w", "c_h"]
coil_a = M2D.modeler.create_rectangle(origin=origin, sizes=sizes, is_covered=True, name="coil_a", material=copper_c, non_model=False)

M2D.modeler.subtract(blank_list=coil_a, tool_list=slot_a, keep_originals=True)  # coil minus slot

vector=["t_w+2*t_g", 0]
M2D.modeler.duplicate_along_line(assignment=coil_a, vector=vector, clones=2, attach=False, is_3d_comp=False, duplicate_assignment=True)
coil_b = M2D.modeler.object_list[-1]
coil_b.name = "coil_b"

M2D.modeler.subtract(blank_list=coil_b, tool_list=slot_b, keep_originals=True)  # coil minus slot

vector=["t_w+2*t_g", 0]
M2D.modeler.duplicate_along_line(assignment=coil_b, vector=vector, clones=2, attach=False, is_3d_comp=False, duplicate_assignment=True)
coil_c = M2D.modeler.object_list[-1]
coil_c.name = "coil_c"

M2D.modeler.subtract(blank_list=coil_c, tool_list=slot_c, keep_originals=True)  # coil minus slot


vector = ["-( 1.5*t_w+2*t_g )", 0]
M2D.modeler.move(assignment=[coil_a, coil_b, coil_c, slot_a, slot_b, slot_c], vector=vector)    # center the slot/coil group


# 5. Stator yoke
origin = ["-( (t_w+2*t_g)*3*1.3 )/2", "c_h/2"]
sizes = ["( (t_w+2*t_g)*3*1.3 )", "cy_th"]
stator = M2D.modeler.create_rectangle(origin=origin, sizes=sizes, is_covered=True, name="stator", material="air", non_model=False)

M2D.modeler.unite(assignment=[stator, slot_a, slot_b, slot_c], purge=False, keep_originals=False)

vector = [0, "my_th/2+m_h+airgap+c_h/2"]
M2D.modeler.move(assignment=[stator, coil_a, coil_b, coil_c], vector=vector)    # place above the magnet array with the air gap


# 6. Line up the stator with the magnet array at its starting alignment
vector = ["-(3*(gap_w+m_w)/2-t_w-2*t_g)", 0]
M2D.modeler.move(assignment=[coil_a, coil_b, coil_c, stator], vector=vector)


# 7. Region -- Maxwell 2D's own create_region() throws an error here, so this uses
# the raw script-recording API instead
oEditor = oDesign.SetActiveEditor("3D Modeler")

oEditor.CreateRegion(
	[
		"NAME:RegionParameters",
		"+XPaddingType:="	, "Absolute Position",
		"+XPadding:="		, "my_w",
		"-XPaddingType:="	, "Absolute Position",
		"-XPadding:="		, "-my_w",
		"+YPaddingType:="	, "Absolute Position",
		"+YPadding:="		, "my_w",
		"-YPaddingType:="	, "Absolute Position",
		"-YPadding:="		, "-my_w",
		"+ZPaddingType:="	, "Percentage Offset",
		"+ZPadding:="		, "0",
		"-ZPaddingType:="	, "Percentage Offset",
		"-ZPadding:="		, "0",
		[
			"NAME:BoxForVirtualObjects",
			[
				"NAME:LowPoint",
				1,
				1,
				1
			],
			[
				"NAME:HighPoint",
				-1,
				-1,
				-1
			]
		]
	],
	[
		"NAME:Attributes",
		"Name:="		, "Region",
		"Flags:="		, "Wireframe#",
		"Color:="		, "(143 175 143)",
		"Transparency:="	, 0,
		"PartCoordinateSystem:=", "Global",
		"UDMId:="		, "",
		"MaterialValue:="	, "\"vacuum\"",
		"SurfaceMaterialValue:=", "\"\"",
		"SolveInside:="		, True,
		"ShellElement:="	, False,
		"ShellElementThickness:=", "nan ",
		"ReferenceTemperature:=", "nan ",
		"IsMaterialEditable:="	, True,
		"IsSurfaceMaterialEditable:=", True,
		"UseMaterialAppearance:=", False,
		"IsLightweight:="	, False
	])

region = M2D.modeler.object_list[-1]


# 8. Balloon boundary on the region so fields radiate outward without reflecting
M2D.assign_balloon(assignment=region.edges, boundary=None)

In [ ]:
# 1. Turn the coil sections into coil boundaries (an in and an out terminal per phase)
M2D.modeler.separate_bodies(assignment=[coil_a], create_group=False)   # split into inner/outer sections
coil_a_out = M2D.modeler.sheet_objects[-1]
coil_a_in = coil_a

M2D.modeler.separate_bodies(assignment=[coil_b], create_group=False)
coil_b_out = M2D.modeler.sheet_objects[-1]
coil_b_in = coil_b

M2D.modeler.separate_bodies(assignment=[coil_c], create_group=False)
coil_c_out = M2D.modeler.sheet_objects[-1]
coil_c_in = coil_c

coil_a_in_ter = M2D.assign_coil(assignment=coil_a_in, conductors_number=turn_c, polarity='Negative', name="coil_a_in_ter")
coil_a_out_ter = M2D.assign_coil(assignment=coil_a_out, conductors_number=turn_c, polarity='Positive', name="coil_a_out_ter")
coil_b_in_ter = M2D.assign_coil(assignment=coil_b_in, conductors_number=turn_c, polarity='Negative', name="coil_b_in_ter")
coil_b_out_ter = M2D.assign_coil(assignment=coil_b_out, conductors_number=turn_c, polarity='Positive', name="coil_b_out_ter")
coil_c_in_ter = M2D.assign_coil(assignment=coil_c_in, conductors_number=turn_c, polarity='Negative', name="coil_c_in_ter")
coil_c_out_ter = M2D.assign_coil(assignment=coil_c_out, conductors_number=turn_c, polarity='Positive', name="coil_c_out_ter")


# 2. Create a winding per phase and add its coils
winding_a = M2D.assign_winding(assignment=None, winding_type='Current', is_solid=False, current=current_a, resistance=0, inductance=0, voltage=0, parallel_branches=1, phase=0, name="winding_a")
winding_b = M2D.assign_winding(assignment=None, winding_type='Current', is_solid=False, current=current_b, resistance=0, inductance=0, voltage=0, parallel_branches=1, phase=0, name="winding_b")
winding_c = M2D.assign_winding(assignment=None, winding_type='Current', is_solid=False, current=current_c, resistance=0, inductance=0, voltage=0, parallel_branches=1, phase=0, name="winding_c")

M2D.add_winding_coils(assignment=winding_a.name, coils=[coil_a_in_ter.name, coil_a_out_ter.name])
M2D.add_winding_coils(assignment=winding_b.name, coils=[coil_b_in_ter.name, coil_b_out_ter.name])
M2D.add_winding_coils(assignment=winding_c.name, coils=[coil_c_in_ter.name, coil_c_out_ter.name])

In [ ]:
# NOTE: MaxLength coarsened from 0.5mm to 4mm, and PercentRefinement dropped
# from 30 to 15 -- at the original settings the mesh crept past the Student
# license's 2D cap during adaptive refinement. AEDT still completes the solve
# in that case (caps the mesh and proceeds with an "Adaptive Passes did not
# converge" warning rather than a hard failure), confirmed locally across all
# 21 sweep points -- this setting is just what was actually verified to run
# cleanly and reasonably quickly.

oModule = oDesign.GetModule("MeshSetup")

oModule.AssignLengthOp(
	[
		"NAME:Length1",
		"RefineInside:="	, False,
		"Enabled:="		, True,
		"Objects:="		, ["magnet_1","magnet_2","magnet_1_Separate1","magnet_1_Separate2","magnet_1_Separate3","magnet_1_Separate4","magnet_2_Separate1","magnet_2_Separate2","magnet_2_Separate3","magnet_2_Separate4","coil_a","coil_b","coil_c","stator","coil_a_Separate1","coil_b_Separate1","coil_c_Separate1"],
		"RestrictElem:="	, False,
		"NumMaxElem:="		, "1000",
		"RestrictLength:="	, True,
		"MaxLength:="		, "4mm",
		"ApplyToInitialMesh:="	, False,
		"IsGlobal:="		, False
	])

In [ ]:
M2D.assign_force(assignment=[coil_a_in, coil_a_out, coil_b_in, coil_b_out, coil_c_in, coil_c_out], coordinate_system='Global', is_virtual=True, force_name="force_virtual")

In [ ]:
## Sweep t_sample through one electrical cycle ##

# Pilot with a coarse step first -- once the per-point solve time is known locally,
# tighten t_step toward "0.001s" to match the original notebook's 100-point resolution.
t_step = "0.005s"

time_sweep = M2D.parametrics.add(variable="t_sample", start_point="0s", end_point="0.1s",
                                  step=t_step, variation_type="LinearStep", name="TimeSweep")

time_sweep.props["ProdOptiSetupDataV2"]["SaveFields"] = True
time_sweep.update()

time_sweep.analyze()

In [ ]:
## Extract force and phase currents across the sweep ##

data = M2D.post.get_solution_data(
    expressions=["force_virtual.Force_x", "force_virtual.Force_y",
                 "InputCurrent(winding_a)", "InputCurrent(winding_b)", "InputCurrent(winding_c)"],
    setup_sweep_name="Setup1 : LastAdaptive",
    primary_sweep_variable="t_sample",
    variations={"t_sample": ["All"]},
)

t_vals = np.array(data.primary_sweep_values, dtype=float)

df = pd.DataFrame({
    "Time [ms]": t_vals * 1000,
    "Force_x [N]": np.array(data.data_real("force_virtual.Force_x"), dtype=float),
    "Force_y [N]": np.array(data.data_real("force_virtual.Force_y"), dtype=float),
    "current_a [A]": np.array(data.data_real("InputCurrent(winding_a)"), dtype=float),
    "current_b [A]": np.array(data.data_real("InputCurrent(winding_b)"), dtype=float),
    "current_c [A]": np.array(data.data_real("InputCurrent(winding_c)"), dtype=float),
})
df = df.sort_values("Time [ms]").reset_index(drop=True)
df

In [ ]:
## Save results to CSV ##

# saved flat, next to the notebook -- same convention as Images/
df.to_csv("Week5_ex3_time_sweep_result.csv", index=False)

In [ ]:
## Plot force vs time ##

os.makedirs("Images", exist_ok=True)

plt.figure(figsize=(12, 8))
plt.plot(df["Time [ms]"], df["Force_x [N]"], marker='o', linestyle='--', linewidth=2, markersize=6, label="Force_x [N]")
plt.plot(df["Time [ms]"], df["Force_y [N]"], marker='o', linestyle='--', linewidth=2, markersize=6, label="Force_y [N]")

plt.title("Force on Coil (Mover Held Fixed)", fontsize=16)
plt.xlabel("Time [ms]", fontsize=16)
plt.ylabel("Force [N]", fontsize=16)
plt.ylim(-4, 4)
plt.legend(fontsize=12)
plt.grid(True, linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.savefig("Images/Week5_ex3_force_plot.png", dpi=150)
plt.show()

In [ ]:
## Plot phase currents vs time ##

plt.figure(figsize=(12, 8))
plt.plot(df["Time [ms]"], df["current_a [A]"], linestyle='-', linewidth=4, label="current_a [A]")
plt.plot(df["Time [ms]"], df["current_b [A]"], linestyle='-', linewidth=4, label="current_b [A]")
plt.plot(df["Time [ms]"], df["current_c [A]"], linestyle='--', linewidth=4, label="current_c [A]")

plt.title("Phase Currents (A Active, B/C Zero)", fontsize=16)
plt.xlabel("Time [ms]", fontsize=16)
plt.ylabel("Current [A]", fontsize=16)
plt.legend(fontsize=12)
plt.grid(True, linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.savefig("Images/Week5_ex3_current_plot.png", dpi=150)
plt.show()

In [ ]:
## Save project ##

M2D.save_project()